# Baselines - Updated

Computes simple baselines and combines Stream A/B/C result tables if available.


In [1]:
from pathlib import Path
import sys, os, json, pickle, re, warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd

def find_repo_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "data").exists() and (q / "src").exists():
            return q
    raise RuntimeError("Could not find repo root.")

ROOT = find_repo_root()
sys.path.insert(0, str(ROOT))

from src.utils.paths import STREAM_C_CACHE, STREAM_A_CACHE, STREAM_B_CACHE, ALL_MANIFEST, FIGURES, TABLES

STREAM_C_CACHE.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("STREAM_C_CACHE:", STREAM_C_CACHE)


ROOT: E:\eve-rag
STREAM_C_CACHE: E:\eve-rag\cache\stream_c


In [2]:
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
df = pd.read_csv(ALL_MANIFEST, low_memory=False)
print(df.shape); display(df.groupby(["dataset", "split"]).size().reset_index(name="rows"))


(97361, 24)


,dataset,split,rows
0,mmfakebench,test,10000
1,mmfakebench,val,1000
2,newsclippings,test,7264
3,newsclippings,train,71072
4,newsclippings,val,7024
5,verite,test,1001


In [3]:
def metric_row(method, dataset_name, y, pred=None, score=None):
    y=np.asarray(y).astype(int)
    if score is not None:
        score=np.asarray(score).astype(float); pred=(score>=0.5).astype(int)
    else: pred=np.asarray(pred).astype(int)
    return {"method":method,"dataset":dataset_name,"acc":round(accuracy_score(y,pred)*100,2),"precision":round(precision_score(y,pred,zero_division=0)*100,2),"recall":round(recall_score(y,pred,zero_division=0)*100,2),"f1":round(f1_score(y,pred,zero_division=0)*100,2),"auroc":round(roc_auc_score(y,score)*100,2) if score is not None and len(np.unique(y))>1 else np.nan}
eval_sets={"newsclippings_test":df[(df["dataset"]=="newsclippings")&(df["split"]=="test")],"mmfakebench_test":df[(df["dataset"]=="mmfakebench")&(df["split"]=="test")],"verite_test":df[(df["dataset"]=="verite")&(df["split"]=="test")]}
rows=[]
for name,sub in eval_sets.items():
    y=sub["label"].astype(int).values
    rows.append(metric_row("majority", name, y, pred=np.full_like(y, int(np.mean(y)>=0.5))))
    rows.append(metric_row("always_match_0", name, y, pred=np.zeros_like(y)))
    rows.append(metric_row("always_mismatch_1", name, y, pred=np.ones_like(y)))
baseline=pd.DataFrame(rows); baseline.to_csv(TABLES / "baseline_results.csv", index=False); display(baseline)


,method,dataset,acc,precision,recall,f1,auroc
0,majority,newsclippings_test,50.00,50.00,100.0,66.67,NaN
1,always_match_0,newsclippings_test,50.00,0.00,0.0,0.00,NaN
2,always_mismatch_1,newsclippings_test,50.00,50.00,100.0,66.67,NaN
3,majority,mmfakebench_test,70.00,70.00,100.0,82.35,NaN
4,always_match_0,mmfakebench_test,30.00,0.00,0.0,0.00,NaN
5,always_mismatch_1,mmfakebench_test,70.00,70.00,100.0,82.35,NaN
6,majority,verite_test,66.23,66.23,100.0,79.69,NaN
7,always_match_0,verite_test,33.77,0.00,0.0,0.00,NaN
8,always_mismatch_1,verite_test,66.23,66.23,100.0,79.69,NaN


In [4]:
clip_path = STREAM_B_CACHE / "features" / "clip_features_all.csv"
if clip_path.exists():
    clip=pd.read_csv(clip_path, low_memory=False); rows=[]
    for name, sub_ids in eval_sets.items():
        sub=clip[clip["pair_id"].isin(sub_ids["pair_id"].astype(str))]
        if len(sub):
            y=sub["label"].astype(int).values; score=pd.Series(-sub["clip_cosine"].astype(float).values).rank(pct=True).values
            rows.append(metric_row("clip_cosine_rank", name, y, score=score))
    clip_base=pd.DataFrame(rows); clip_base.to_csv(TABLES / "clip_baseline_results.csv", index=False); display(clip_base)
else: print("No Stream B features found.")


,method,dataset,acc,precision,recall,f1,auroc
0,clip_cosine_rank,newsclippings_test,65.74,65.73,65.75,65.74,71.97
1,clip_cosine_rank,mmfakebench_test,58.13,78.12,55.81,65.11,64.13
2,clip_cosine_rank,verite_test,48.25,64.47,48.72,55.50,48.54


In [5]:
tables=[]
for fname,tag in [("stream_a_results.csv","stream_a"),("stream_b_results.csv","stream_b"),("stream_c_results.csv","stream_c"),("baseline_results.csv","baseline"),("clip_baseline_results.csv","clip_baseline")]:
    p=TABLES/fname
    if p.exists():
        t=pd.read_csv(p); t["source_table"]=tag; tables.append(t)
combined=pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()
combined.to_csv(TABLES / "all_streams_and_baselines_summary.csv", index=False)
display(combined)
print("Baselines complete.")


,method,dataset,acc,precision,recall,f1,auroc,source_table
0,stream_a_lr,mmfakebench_test,62.56,78.19,64.51,70.70,65.41,stream_a
1,majority_train,mmfakebench_test,70.00,70.00,100.00,82.35,NaN,stream_a
2,caption_clip_sim_median,mmfakebench_test,58.10,78.10,55.79,65.08,64.14,stream_a
3,stream_a_lr,verite_test_local,53.30,69.04,54.00,60.60,53.83,stream_a
4,majority_train,verite_test_local,66.51,66.51,100.00,79.88,NaN,stream_a
5,caption_clip_sim_median,verite_test_local,48.47,64.95,48.91,55.80,47.82,stream_a
6,NaN,newsclippings_val,66.22,67.97,61.33,64.48,73.42,stream_b
7,NaN,newsclippings_test,65.87,67.60,60.96,64.11,73.91,stream_b
8,NaN,mmfakebench_test,46.95,82.38,30.80,44.84,63.35,stream_b
9,NaN,verite_test_local,63.77,66.32,92.49,77.25,49.12,stream_b


Baselines complete.
